In [1]:
import re
import numpy as np
import pandas as pd
from collections import Counter

In [3]:
CSV_PATH = "inputs/discharge_filtered.csv"
TEXT_COL = "text"

df = pd.read_csv(CSV_PATH)
df = df[df[TEXT_COL].notna()].copy()


SECTION_HEADER_RE = re.compile(r"(?im)^\s*([A-Za-z][A-Za-z0-9 /&\-\(\)\#]+)\s*:\s*$") #It only matches if the “Title:” is alone on the line (because of ^ and $) and uses multi-line mode.
HEADER_BLOCK_RE   = re.compile(r"(?is)\bname:\s*.*?\bchief complaint:\s*") 
#Removes the de-identified “header” part of the note (name/unit/admission date/etc.)
#It deletes everything starting at name: up to the first chief complaint:.


KEEP_PREFIXES = [ #These are sections we want because they often contain meaningful narrative info:
    "chief complaint","history of present illness","brief hospital course",
    "hospital course","hospital course by problem","active issues",
    "chronic issues","transition of care","studies","imaging",
    "other studies","discharge diagnosis","diagnosis",
    "major surgical or invasive procedure",
]

DROP_PREFIXES = [ #These are sections we don’t want because they often create repetitive boilerplate or noise:
    "physical exam","admission physical exam","discharge physical exam",
    "pertinent results","admission labs","discharge labs",
    "interval/discharge labs","micro","pathology",
    "medications on admission","discharge medications",
    "discharge instructions","followup instructions",
    "discharge disposition","facility","discharge condition",
    "social history","family history","review of systems","allergies",
]

def split_sections(text):
    matches = list(SECTION_HEADER_RE.finditer(text))
    if not matches:
        return [("full_text", text)]
    blocks = []
    for i, m in enumerate(matches):
        title = m.group(1).strip().lower() #matched title
        start = m.end() #content: text from end of this header 
        end = matches[i+1].start() if i+1 < len(matches) else len(text) #until the next header
        blocks.append((title, text[start:end].strip())) 
    return blocks #What it returns: A list of (section_title, section_content).

def keep_useful_sections(text):
    blocks = split_sections(text)
    if len(blocks)==1 and blocks[0][0]=="full_text": 
        return text # If section splitting failed (only “full_text”), keep the whole note (fallback).
    kept=[]
    for title,content in blocks:
        if any(title.startswith(d) for d in DROP_PREFIXES):
            continue #If its title starts with something in DROP_PREFIXES → skip it.
        if any(title.startswith(k) for k in KEEP_PREFIXES):
            kept.append(content)#If its title starts with something in KEEP_PREFIXES → keep the content.
    return "\n".join(kept) if kept else text

TEMPLATE_PHRASES = [
    r"\bhistory of present illness\b",
    r"\bbrief hospital course\b",
    r"\bhospital course\b",
    r"\bdischarge diagnosis\b",
    r"\bchief complaint\b",
]

def clean_note(text):
    if not isinstance(text,str):
        return ""
    text = re.sub(HEADER_BLOCK_RE," ",text) # Remove admin header
    text = text.replace("___"," ") #Remove de-identification placeholders
    text = re.sub(r"\[\*\*.*?\*\*\]"," ",text) #Remove de-identification placeholders
    text = keep_useful_sections(text) #text = keep_useful_sections(text)
    text = text.lower()
    for pat in TEMPLATE_PHRASES: #Remove common template phrases
        text = re.sub(pat," ",text)
    text = re.sub(r"=+|_+"," ",text) #Remove divider lines
    text = re.sub(r"\s+"," ",text).strip() #Collapse whitespace
    return text

df["text_clean"] = df[TEXT_COL].apply(clean_note)
#Result: text_clean = cleaned, lowercased narrative text with less boilerplate.

TOKEN_RE = re.compile(r"[a-z]+") #Keeps only sequences of letters (no numbers, no punctuation).

STOPWORDS = set("""
a an the and or but if then else to of in on for with without by from at as is are was were be been being
this that these those it its patient pt she he they them his her their
mg ml mcg hr hrs day days week weeks month months year years yo old female male
had have has not no which who also likely found showed present prior during after
given started continued placed noted received transferred arrival initial report decision made
""".split()) #Removes common English words + common medical-note filler words (pt, mg, day, etc.).

def tokenize(t):
    toks = TOKEN_RE.findall(t)
    return [x for x in toks if x not in STOPWORDS and len(x)>1]
    #Extracts words, Drops stopwords, Drops 1-letter tokens

token_lists = [tokenize(t) for t in df["text_clean"].tolist()] #Creates a list of token lists (one per note).
N = len(token_lists) #number of notes.

def make_ngrams(tokens,n): #Make n-grams from tokens
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def top_ngrams(token_lists, n, topk, min_count):
    """
    Counts occurrences of each n-gram across the dataset (raw frequency).
    Filters out rare ones (min_count).
    Returns the topk most common.
    """
    c=Counter()
    for toks in token_lists:
        if len(toks) >= n:
            c.update(make_ngrams(toks,n))
    items=[(k,v) for k,v in c.items() if v>=min_count]
    items.sort(key=lambda x:x[1],reverse=True)
    return items[:topk]

def doc_freq(token_lists,phrase):
    """
    Counts in how many notes the phrase appears at least once.
    Uses set(...) so multiple repeats in one note count only once.
    """
    n=len(phrase.split())
    cnt=0
    for toks in token_lists:
        if len(toks)>=n and phrase in set(make_ngrams(toks,n)):
            cnt+=1
    return cnt


top_uni = top_ngrams(token_lists,1,50,120)
top_bi  = top_ngrams(token_lists,2,60,40)
top_tri = top_ngrams(token_lists,3,40,25)

print("\nTOP BIGRAMS (doc freq)")
for p,c in top_bi:
    print(f"{doc_freq(token_lists,p):>5}/{N} | {c:>4} | {p}")

#doc_freq/N = number of notes containing it
#c = total raw occurrences across all notes
#p = the phrase

print("\nTOP TRIGRAMS (doc freq)")
for p,c in top_tri:
    print(f"{doc_freq(token_lists,p):>5}/{N} | {c:>4} | {p}")

# =========================
# Indicator dictionary
# =========================
INDICATORS = {
    "Anoxic brain injury": r"\b(anoxic brain|hypoxic ischemic encephal)\w*",
    "Seizure": r"\b(seizure|myoclon)\w*",
    "EEG": r"\b(eeg|electroencephalogram)\b",
    "Coma": r"\b(coma|comatose|unresponsive|unarousable)\b",

    "Vasopressor": r"\b(norepi|levophed|vasopressin|pressor|pressors)\b",
    "Shock": r"\b(cardiogenic shock|septic shock|shock)\b",
    "Hypotension": r"\b(hypotension|sbp\s*<\s*90)\b",

    "TTM": r"\b(ttm|targeted temperature|hypothermia|cooling)\b",
    "ROSC": r"\b(return of spontaneous circulation|rosc)\b",

    "Myocardial infarction": r"\b(stemi|nstemi|myocardial infarct|acute mi)\b",
    "Cardiac cath": r"\b(cardiac cath|catheterization|pci|stent)\b",
    "Arrhythmia": r"\b(vf arrest|vt arrest|arrhythmia|atrial fibrillation)\b",

    "Respiratory failure": r"\b(respiratory failure|hypoxemic respiratory|hypercapnic respiratory)\b",
    "Mechanical ventilation": r"\b(intubated|mechanical ventilation|ventilator)\b",
    "ARDS": r"\b(ards|acute respiratory distress)\b",
    "Pulmonary edema": r"\b(pulmonary edema)\b",

    "AKI": r"\b(aki|acute kidney injury|acute renal failure)\b",
    "Dialysis": r"\b(crrt|dialysis|hemodialysis)\b",

    "Metabolic acidosis": r"\b(metabolic acidosis|lactic acidosis|anion gap)\b",
    "Lactate": r"\b(lactate)\b",

    "Sepsis": r"\b(sepsis|septic)\b",
    "Pneumonia": r"\b(pneumonia|aspiration)\b",

    "DNR/DNI": r"\b(dnr|dni|do not resuscitate|do not intubate)\b",
    "Comfort care": r"\b(comfort measures|cmo|withdrawal of care|palliative)\b",
    "Death": r"\b(expired|pronounced dead|death)\b",
}

# detect
def detect(text,pat):
    """
    Searches case-insensitively in the cleaned text.
    Returns 1 if found, 0 if not.
    """
    return int(bool(re.search(pat,text,flags=re.I)))

for name,pat in INDICATORS.items():#Adds one new column per indicator, containing 0/1 per note.
    df[name] = df["text_clean"].apply(lambda x: detect(x,pat))

rows=[]
for k in INDICATORS:
    cnt=df[k].sum() #cnt = number of notes where it was mentioned (since column is 0/1).
    rows.append((k,cnt,cnt/N)) #cnt/N = fraction of notes mentioning it.

prev=pd.DataFrame(rows,columns=["Indicator","Num_notes","Frac_notes"]).reset_index(drop=True)
prev=prev.sort_values("Num_notes",ascending=False)

print("\nINDICATOR PREVALENCE")
print(prev)

FileNotFoundError: [Errno 2] No such file or directory: 'inputs/discharge_filtered.csv'